# CSC 4792 Group 36 Samfya Town Council Data Project

## Web extraction cleaning and curation notebook

**Assigned council:** Samfya Town Council, Zambia  
**Official source:** https://www.samfyacouncil.gov.zm  
**Course:** CSC 4792 Data Mining and Warehousing

This notebook documents the reproducible workflow used to collect public council records, validate raw outputs, clean and standardise the data, and review the final Kaggle-ready CSV files. All dataset files use the pipe character (`|`) as the separator required by the assignment.


## Group 36 responsibilities

The roles below describe the work reported by the group:

| Member | Contribution |
|---|---|
| Gift Kanene | Group coordination, web scraping, software, data curation, GitHub and Moodle coordination |
| Khadijah Zimba | Support for data collection and submission coordination |
| Ben Samuel | Data cleaning, validation, and documentation |
| Mwansa Matanda | Kaggle dataset description and publication coordination |
| Lusungu Mulenga | Data description paper preparation |

The notebook focuses on the technical workflow. The Kaggle dataset and the Data Description Paper are linked in the final section.


## Assignment requirements addressed here

The CSC 4792 brief requires a functional codebase for web scraping, data extraction, cleaning, and preprocessing. It also requires that every dataset-creation step is documented in Markdown cells and that the notebook is committed to GitHub and submitted as an `.ipynb` file.

This notebook records:

1. the public Samfya Town Council sources and target record types;
2. the scraping and document-discovery workflow;
3. raw-data validation before cleaning;
4. cleaning and preprocessing rules; and
5. checks of the three final pipe-separated CSV files.


## Data scope

The project collects publicly available material from Samfya Town Council webpages and linked documents. The dataset covers Constituency Development Fund projects and activities, council documents and resources, and council news. Source URLs and retrieval information are retained for traceability.

The final cleaned outputs are:

- `db-unza26-csc4792-samfya_cdf_projects.csv` - 28 records, 30 columns
- `db-unza26-csc4792-samfya_council_resources.csv` - 90 records, 17 columns
- `db-unza26-csc4792-samfya_news.csv` - 21 records, 20 columns


## Reproducible project setup

Run this notebook from the repository's `notebooks` folder, or open it in Google Colab after preserving the repository folder layout. To recreate the workflow locally:

```text
python -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
python scripts/run_all.py
python scripts/validate_raw_data.py
python scripts/clean_data.py
```

Live scraping is deliberately not run automatically below. It accesses the council website and should be run only when a fresh collection is required. The existing cleaned files are loaded for review in this notebook.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the repository whether the notebook is opened from its own folder or the repository root.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "data").exists():
    ROOT = next(path for path in ROOT.parents if (path / "data").exists())

RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "cleaned"
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

print(f"Project root: {ROOT}")
print(f"Raw-data folder: {RAW}")
print(f"Cleaned-data folder: {CLEAN}")


## 1 Data extraction and document discovery

The `samfya_scraper.py` module discovers public council pages, indexes linked documents, and collects paginated news posts. It uses a descriptive user agent, timeouts, retries, URL deduplication, and a low request rate. The scraper targets CDF and community-project pages, grants and loans, skills and bursaries, council publications, and news pages.

The import below confirms the reusable scraper interface. The next cell is intentionally commented out so opening the notebook does not start live web requests.


In [ ]:
from samfya_scraper import ScrapeConfig, run as run_scraper

config = ScrapeConfig(
    delay_seconds=1.0,
    max_news_pages=10,
    download_documents=False,
    fetch_article_details=True,
)
print("Scraper configuration prepared. Run the next cell only for a fresh collection.")


In [ ]:
# Fresh collection command. This writes raw pipe-separated files to data/raw/.
# resources, news = run_scraper(config)
# display(resources.head())
# display(news.head())


### Optional document download and PDF extraction

The resource index is reviewed before downloading linked files. Where needed, public documents can be downloaded at the same low request rate and processed with the PDF extraction helper.

```text
pip install -r requirements-pdf.txt
python scripts/extract_pdf_tables.py
```

The extractor writes raw page text, extracted tables, a manifest, and CDF evidence references. These raw artefacts are reviewed before values are standardised for the curated dataset.


## 2 Raw-data validation

Before cleaning, `validate_raw_data.py` checks that expected raw files exist, have a header and data rows, use the pipe delimiter, include required columns, and retain source URLs. It also reports repeated URLs for resource and news indexes as warnings for review.


In [ ]:
# Run after a fresh scrape. The validation script writes data/raw/validation_report.csv.
# from validate_raw_data import run as validate_raw_data
# validation_exit_code = validate_raw_data()
# print("Validation exit code:", validation_exit_code)


## 3 Data cleaning and preprocessing

`clean_data.py` reads the raw pipe-separated files without changing them in place and writes the final CSV files to `data/cleaned/`. The pipeline:

- trims and normalises whitespace;
- standardises the council name and known ward names;
- maps project-status terms to a controlled vocabulary;
- normalises dates to ISO format and extracts four-digit years;
- retains original monetary text and parses only unambiguous Zambian Kwacha amounts;
- removes exact duplicate rows; and
- preserves source URLs and retrieval fields for provenance.


In [ ]:
from clean_data import main as clean_dataset

print("Cleaning pipeline imported.")
print("To recreate the cleaned files from current raw files, run: clean_dataset()")

# Uncomment only when regenerating the three CSV outputs from data/raw/.
# clean_dataset()


## 4 Load and validate the final curated dataset

The following cells load the same three cleaned CSV files published on Kaggle. They verify the required delimiter, record totals, column totals, absence of exported index columns, and presence of the relevant source URLs.


In [ ]:
FILES = {
    "CDF projects and activities": "db-unza26-csc4792-samfya_cdf_projects.csv",
    "Council resources": "db-unza26-csc4792-samfya_council_resources.csv",
    "Council news": "db-unza26-csc4792-samfya_news.csv",
}

datasets = {
    label: pd.read_csv(CLEAN / filename, sep="|", dtype=str, keep_default_na=False)
    for label, filename in FILES.items()
}
cdf = datasets["CDF projects and activities"]
resources = datasets["Council resources"]
news = datasets["Council news"]

summary = pd.DataFrame(
    [
        {
            "Dataset": label,
            "File": FILES[label],
            "Records": len(frame),
            "Columns": len(frame.columns),
        }
        for label, frame in datasets.items()
    ]
)
summary


In [ ]:
expected_shape = {
    "CDF projects and activities": (28, 30),
    "Council resources": (90, 17),
    "Council news": (21, 20),
}

checks = []
for label, frame in datasets.items():
    filename = CLEAN / FILES[label]
    first_line = filename.read_text(encoding="utf-8-sig").splitlines()[0]
    checks.append({
        "Dataset": label,
        "Expected shape": expected_shape[label],
        "Actual shape": frame.shape,
        "Pipe delimiter": "|" in first_line,
        "No exported index": not any(col.lower().startswith("unnamed") for col in frame.columns),
        "Shape correct": frame.shape == expected_shape[label],
    })

checks = pd.DataFrame(checks)
assert checks["Pipe delimiter"].all()
assert checks["No exported index"].all()
assert checks["Shape correct"].all()
checks


In [ ]:
provenance_checks = pd.DataFrame({
    "Dataset": ["CDF projects and activities", "Council resources", "Council news"],
    "Source field": ["source_url_clean", "document_url_clean", "article_url_clean"],
    "Missing source URLs": [
        (cdf["source_url_clean"] == "").sum(),
        (resources["document_url_clean"] == "").sum(),
        (news["article_url_clean"] == "").sum(),
    ],
})
assert (provenance_checks["Missing source URLs"] == 0).all()
provenance_checks


## 5 Review of cleaned fields

These previews demonstrate the standardised project, resource, and news fields that users can analyse. The full source URLs remain in the files so individual records can be traced to the public source material.


In [ ]:
cdf[[
    "record_id", "record_kind", "council_clean", "ward_clean",
    "project_name_clean", "status_clean", "year_clean",
    "amount_zmw_raw_kept", "amount_zmw", "source_url_clean",
]].head(10)


In [ ]:
cdf_amounts = pd.to_numeric(cdf["amount_zmw"], errors="coerce")
print("Rows with a numeric amount:", cdf_amounts.notna().sum(), "of", len(cdf))
print("Sum of unambiguous numeric amounts:", f"{cdf_amounts.sum():,.2f} ZMW")
print("\nProject and activity status distribution:")
print(cdf["status_clean"].value_counts(dropna=False).to_string())


In [ ]:
resources[[
    "record_id", "category_clean", "year_clean", "document_title_clean",
    "file_type_clean", "document_url_clean",
]].head(10)


In [ ]:
news[[
    "record_id", "title_clean", "published_date_clean",
    "mentioned_amount_zmw", "article_url_clean",
]].head(10)


## 6 Published outputs and reproducibility

**Kaggle dataset:** https://www.kaggle.com/datasets/giftkanene/samfya-council-development-records  
**GitHub repository:** https://github.com/Giftkanene/group36-data-mining-project

The Kaggle page contains the three final cleaned pipe-separated CSV files. This repository contains the scraper, raw-data validator, cleaner, PDF extraction helper, cleaned outputs, and this notebook. To reproduce or extend the dataset, run a fresh collection, review the raw validation report, rerun the cleaning pipeline, and repeat the checks in Section 4 before publishing new files.
